# TGA kaynak kabul kontrolü
Proje kökünde kernel başlatın. Yalnızca başlık incelenir; ikili ölçüm verisi çözülmez. Notebook bu teslimde çalıştırılmadı; eşdeğer CLI kontrolü çalıştırıldı.

In [ ]:
import json, zipfile, hashlib
from pathlib import Path
from pipelines.ingestion.tga_inspect import inspect_header
manifest = json.loads(Path('data/manifests/tga-zenodo-17659041.json').read_text(encoding='utf-8'))
entry = manifest['files'][1]
archive = Path('storage/tga/17659041') / entry['raw_path']
assert hashlib.sha256(archive.read_bytes()).hexdigest() == entry['sha256']
with zipfile.ZipFile(archive) as z:
    results = [inspect_header(z.read(i)) for i in z.infolist() if not i.is_dir() and i.file_size <= 1024*1024]
{'received': len(results), 'quarantined': sum(r['status']=='QUARANTINED' for r in results), 'duplicates': len(results)-len({r['sha256'] for r in results})}